# Generate template stimulus waveforms

Define and save the three standard stimuli to `data/template_waveforms.pkl`.
Tune the parameters in each cell, run **Plot all** to preview, then run **Save** when happy.

| Name | Description |
|---|---|
| `pulse_25ms` | 20 ms silence then 25 ms square pulse |
| `pulse_135ms` | 20 ms silence then 135 ms square pulse |
| `train_25hz_1s` | 20 ms silence then 1 s of 20 ms pulses at 25 Hz |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
from pathlib import Path

from designer_waveform.waveforms import RectangularPulseWaveform, PulseTrainWaveform

## Waveform 1 — short square pulse (25 ms)

In [ ]:
ONSET_MS_1    = 20.0
DURATION_MS_1 = 25.0
AMPLITUDE_1   = 1.0
# Recommended stim window: onset + duration + response tail
STIM_DUR_MS_1 = 120.0

wf_pulse_25 = RectangularPulseWaveform(
    onset_ms    = ONSET_MS_1,
    duration_ms = DURATION_MS_1,
    amplitude   = AMPLITUDE_1,
)
print(wf_pulse_25)

## Waveform 2 — long square pulse (135 ms)

In [ ]:
ONSET_MS_2    = 20.0
DURATION_MS_2 = 135.0
AMPLITUDE_2   = 1.0
STIM_DUR_MS_2 = 250.0

wf_pulse_135 = RectangularPulseWaveform(
    onset_ms    = ONSET_MS_2,
    duration_ms = DURATION_MS_2,
    amplitude   = AMPLITUDE_2,
)
print(wf_pulse_135)

## Waveform 3 — 25 Hz pulse train (1 s)

20 ms pulses at 25 Hz → 20 ms on, 20 ms off (50% duty cycle, 40 ms period).

In [ ]:
ONSET_MS_3         = 20.0
PULSE_DURATION_MS  = 20.0
FREQUENCY_HZ       = 25.0
TRAIN_DURATION_MS  = 1000.0
AMPLITUDE_3        = 1.0
STIM_DUR_MS_3      = ONSET_MS_3 + TRAIN_DURATION_MS + 100.0  # 1120 ms total

wf_train_25hz = PulseTrainWaveform(
    onset_ms          = ONSET_MS_3,
    pulse_duration_ms = PULSE_DURATION_MS,
    frequency_hz      = FREQUENCY_HZ,
    train_duration_ms = TRAIN_DURATION_MS,
    amplitude         = AMPLITUDE_3,
)
print(wf_train_25hz)
period_ms = 1000.0 / FREQUENCY_HZ
print(f'Period: {period_ms:.0f} ms  |  duty cycle: {PULSE_DURATION_MS/period_ms*100:.0f}%')

## Plot all three waveforms

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3))

_templates = [
    ('pulse_25ms',    wf_pulse_25,   STIM_DUR_MS_1, 'tab:blue'),
    ('pulse_135ms',   wf_pulse_135,  STIM_DUR_MS_2, 'tab:orange'),
    ('train_25hz_1s', wf_train_25hz, STIM_DUR_MS_3, 'tab:green'),
]

for ax, (name, wf, dur, color) in zip(axes, _templates):
    t = np.linspace(0, dur, int(dur / 0.1))  # 0.1 ms resolution for plotting
    ax.fill_between(t, 0, wf(t), color=color, alpha=0.4)
    ax.plot(t, wf(t), color=color, lw=1.5)
    ax.set_xlim(0, dur)
    ax.set_ylim(-0.05, 1.15)
    ax.set_xlabel('Time from stim onset (ms)')
    ax.set_ylabel('Envelope amplitude')
    ax.set_title(name)
    ax.spines[['top', 'right']].set_visible(False)

fig.tight_layout()
plt.show()

## Save to data/

In [ ]:
DATA_DIR = Path('../data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

templates = {
    'pulse_25ms': {
        'waveform':     wf_pulse_25,
        'stim_dur_ms':  STIM_DUR_MS_1,
        'description':  f'{ONSET_MS_1:.0f} ms silence + {DURATION_MS_1:.0f} ms square pulse',
    },
    'pulse_135ms': {
        'waveform':     wf_pulse_135,
        'stim_dur_ms':  STIM_DUR_MS_2,
        'description':  f'{ONSET_MS_2:.0f} ms silence + {DURATION_MS_2:.0f} ms square pulse',
    },
    'train_25hz_1s': {
        'waveform':     wf_train_25hz,
        'stim_dur_ms':  STIM_DUR_MS_3,
        'description':  (f'{ONSET_MS_3:.0f} ms silence + {TRAIN_DURATION_MS:.0f} ms train '
                         f'({PULSE_DURATION_MS:.0f} ms pulses at {FREQUENCY_HZ:.0f} Hz)'),
    },
}

SAVE_PATH = DATA_DIR / 'template_waveforms.pkl'
with open(SAVE_PATH, 'wb') as f:
    pickle.dump(templates, f)

print(f'Saved to {SAVE_PATH}')
for name, t in templates.items():
    print(f'  {name:<18} : {t["description"]}  (stim_dur={t["stim_dur_ms"]:.0f} ms)')